# Catbird Intel: real teachers → Ettin-68M, v1

This notebook runs real pinned Hugging Face models on real exported posts. No regex value is a teacher target. No quality, speed or credit metric is fabricated. Use a separate **A100 (40GB+) or H100 GPU** runtime; never attach the archive downloader. All outputs/checkpoints persist on Drive. Read `colab-v1/README_REAL_MODELS.md` before launching.

Cells 1–5 can run before graph rebuild completion. Cells 6–8 require the actual v1 graph pair bank. Full training never consumes the human-review holdout.


## Cell 1 — Mount Drive in this separate GPU runtime

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
LAKE = Path('/content/drive/MyDrive/CatbirdCommunity/catbird-lake')
SOURCE = LAKE / 'colab-v1'
assert (SOURCE / 'real_pipeline.py').exists(), f'Missing uploaded source: {SOURCE}'


## Cell 2 — Install real model dependencies
If Colab requests a restart, restart and rerun cells 1 and 3. Installation is not a completed training run.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(SOURCE / 'requirements-colab.txt')], check=True)


## Cell 3 — Verify runtime and source data; choose Qwen or explicit NLI fallback

In [ ]:
import sys, json, hashlib, subprocess
import torch
sys.path.insert(0, str(SOURCE))
from real_pipeline.common import sha256, read_rows, validate_posts, require_disjoint
assert torch.cuda.is_available(), 'A separate Colab GPU runtime is required'
print({'gpu': torch.cuda.get_device_name(0), 'memory_bytes': torch.cuda.get_device_properties(0).total_memory})
BEHAVIOR_TEACHER = 'qwen'  # Explicit fallback: 'nli'. Never silently substitute.
assert BEHAVIOR_TEACHER in ('qwen', 'nli')
if BEHAVIOR_TEACHER == 'qwen':
    assert torch.cuda.get_device_properties(0).total_memory >= 38_000_000_000, 'Choose A100/H100 or explicit nli fallback'
BANK = LAKE / 'training-banks/phase3_health_training_150k_v1_unlabeled.parquet'
HOLDOUT = LAKE / 'eval/phase3_holdout_600_unlabeled.jsonl'
PAIRS = LAKE / 'training-banks/phase5_social_encoder_pairs_v1.parquet'
RUN = LAKE / f'runs/real-v1-{BEHAVIOR_TEACHER}'
RUN.mkdir(parents=True, exist_ok=True)
source_manifest = json.loads((SOURCE / 'source_manifest.json').read_text())
for relative, digest in source_manifest['files'].items():
    assert sha256(SOURCE / relative) == digest, f'Implementation source changed: {relative}'
print({'implementation_source_manifest': sha256(SOURCE / 'source_manifest.json')})
manifest = json.loads(BANK.with_suffix('.manifest.json').read_text())
assert sha256(BANK) == manifest['sha256'], 'Bank digest mismatch'
holdout_manifest = json.loads((LAKE / 'eval/phase3_v1_export_manifest.json').read_text())
# The sealed holdout has a separately exported .sha256 file.
assert sha256(HOLDOUT) == (HOLDOUT.parent / (HOLDOUT.name + '.sha256')).read_text().split()[0]
assert sha256(HOLDOUT) == holdout_manifest['holdout']['sha256']
assert sha256(BANK) == holdout_manifest['bank']['sha256']
train_rows, holdout_rows = read_rows(BANK), read_rows(HOLDOUT)
assert all(r.get('author_split_bucket') in range(1,20) for r in train_rows), 'Reserved author bucket leaked into train'
assert all(r.get('author_split_bucket') == 0 for r in holdout_rows), 'Holdout author bucket incorrect'
validate_posts(train_rows); validate_posts(holdout_rows); require_disjoint(train_rows, holdout_rows)
print({'bank_rows': len(train_rows), 'holdout_rows': len(holdout_rows), 'bank_sha256': sha256(BANK), 'holdout_sha256': sha256(HOLDOUT), 'outputs': str(RUN)})
# Subprocesses release all GPU allocations between stages. Failures stop the cell.
def run(*args):
    subprocess.run([sys.executable, '-u', str(SOURCE / 'real_pipeline.py'), *map(str,args)], check=True)


## Cell 4 — Label real training posts and the held-out review set
Durable shards resume with unchanged input/model/rubric identity. Qwen uses single-token 0–9 log probabilities, not generated prose. NLI fallback identity is retained. No human quality metric is computed.

In [ ]:
for name, source, split in [('train', BANK, 'train'), ('holdout', HOLDOUT, 'holdout')]:
    common = ['--input', source, '--holdout', HOLDOUT, '--split', split, '--mode', 'colab', '--device', 'cuda']
    run('label', *common, '--teacher', 'political', '--max-length', 512, '--batch-size', 8, '--output', RUN / f'{name}-political')
    run('label', *common, '--teacher', BEHAVIOR_TEACHER, '--max-length', 1024 if BEHAVIOR_TEACHER == 'qwen' else 512, '--batch-size', 8, '--output', RUN / f'{name}-{BEHAVIOR_TEACHER}')
    run('merge', *common, '--political', RUN / f'{name}-political', '--behavior', RUN / f'{name}-{BEHAVIOR_TEACHER}', '--behavior-teacher', BEHAVIOR_TEACHER, '--output', RUN / f'{name}-labeled.parquet')


## Cell 5 — Prepare Josh’s human review sheet
Human labels remain null until Josh reviews them. This pipeline never locks thresholds or reports F1 against its own teacher outputs.

In [ ]:
run('review-sheet', '--input', RUN / 'holdout-labeled.parquet', '--output', RUN / 'holdout-human-review.jsonl', '--split', 'holdout', '--mode', 'colab', '--device', 'cuda')
print(RUN / 'holdout-human-review.jsonl')


## Cell 6 — Require the actual archive-only v1 pair bank
A missing bank is an honest stop. Do not substitute the invalidated v0 bank. These are weak graph-membership positives, not verified co-engagement.

In [ ]:
from real_pipeline.common import validate_pairs
assert PAIRS.exists(), f'v1 graph rebuild/pair export still required: {PAIRS}'
pairs, pair_manifest = validate_pairs(PAIRS, HOLDOUT)
print({'pairs': len(pairs), 'graph_version': pair_manifest['graph_version'], 'known_for_sha256': pair_manifest['known_for_sha256'], 'pair_sha256': sha256(PAIRS)})


## Cell 7 — Train the two heads and export int8 ONNX
5,000 steps is a provisional schedule, not a quality guarantee. Checkpoints persist every 250 steps. To resume, set RESUME to the checkpoint path and leave all run settings unchanged.

In [ ]:
RESUME = None  # e.g. RUN / 'student/checkpoint-0000250.pt'
extra = ['--resume', RESUME] if RESUME else []
run('train', '--input', RUN / 'train-labeled.parquet', '--holdout', HOLDOUT, '--pairs', PAIRS, '--mode', 'colab', '--device', 'cuda', '--steps', 5000, '--batch-size', 32, '--save-every', 250, '--max-length', 256, '--output', RUN / 'student', *extra)


## Cell 8 — Inspect measured receipts and stop the GPU
File sizes and numerical parity are measured. A numerical export pass is not a human quality pass. Credit use remains unknown unless separately read from the actual Colab account. Disconnect/delete this runtime after saving outputs; leave the downloader runtime alone.

In [ ]:
for name in ['training_receipt.json', 'onnx_receipt.json']:
    print(name)
    print((RUN / 'student' / name).read_text())
print('Human review remains required; no production deployment or threshold lock occurred.')
